In [ ]:
pip install scikit-learn pandas matplotlib seaborn numpy --break-system-packages -q 2>&1 | tail -3

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import matplotlib.patches as mpatches
from sklearn.datasets import fetch_openml
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.decomposition import PCA
import warnings
import os  # <-- TAMBAHAN: Untuk membuat direktori otomatis

warnings.filterwarnings('ignore')

# ── 1. LOAD DATASET ──────────────────────────────────────────────────────────
heart = fetch_openml(name='heart-statlog', version=1, as_frame=True, parser='auto')
df = heart.frame.copy()
df.columns = [c.strip() for c in df.columns]

# The target column varies by version; detect it
target_col = [c for c in df.columns if 'class' in c.lower() or c == 'target']
if target_col:
    target_col = target_col[0]
else:
    target_col = df.columns[-1]

# Encode target: 'present'/'absent' or 1/2 → 1/0
target_raw = df[target_col]
if target_raw.dtype == object or str(target_raw.dtype) == 'category':
    cats = target_raw.astype(str).str.strip().str.lower()
    y_true = (cats == 'present').astype(int).values
else:
    y_true = (target_raw.astype(int) > 0).astype(int).values  # 2→1, 1→0

# Feature matrix
X_raw = df.drop(columns=[target_col]).apply(pd.to_numeric, errors='coerce').fillna(0)
feature_names = X_raw.columns.tolist()

scaler = StandardScaler()
X = scaler.fit_transform(X_raw)

# PCA for 2D projection
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X)

# ── 2. ELBOW + SILHOUETTE (k=2..10) ─────────────────────────────────────────
K_range = range(2, 11)
inertias, sil_scores = [], []
for k in K_range:
    km = KMeans(n_clusters=k, n_init=20, random_state=42)
    km.fit(X)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X, km.labels_))

best_k_sil = list(K_range)[np.argmax(sil_scores)]

# ── 3. K=2 analysis ─────────────────────────────────────────────────────────
km2 = KMeans(n_clusters=2, n_init=30, random_state=42)
labels2 = km2.fit_predict(X)

# Align cluster labels to majority class
from scipy.stats import mode
def align_labels(labels, y_true):
    n_clusters = len(np.unique(labels))
    new_labels = labels.copy()
    for c in range(n_clusters):
        mask = labels == c
        if mask.sum() == 0: continue
        dominant = mode(y_true[mask], keepdims=True).mode[0]
        new_labels[mask] = dominant
    return new_labels

labels2_aligned = align_labels(labels2, y_true)

# Confusion-style mixing table
from collections import Counter
mix = pd.crosstab(
    pd.Series(labels2, name='Cluster'),
    pd.Series(y_true, name='True Label')
)
# Rename
mix.columns = ['True:Sehat(0)', 'True:Sakit(1)']
mix.index = [f'Cluster {i}' for i in mix.index]

# Purity per cluster
purity = []
for i in range(2):
    mask = labels2 == i
    cnt = Counter(y_true[mask])
    dom = max(cnt.values())
    purity.append(dom / mask.sum() * 100)

# ── 4. K=4 sub-cluster analysis ─────────────────────────────────────────────
km4 = KMeans(n_clusters=4, n_init=30, random_state=42)
labels4 = km4.fit_predict(X)

# Assign k4 sub-clusters to k2 parent clusters
sub_map = {}
for c4 in range(4):
    mask = labels4 == c4
    parent = mode(labels2[mask], keepdims=True).mode[0]
    sub_map[c4] = parent

# Build detailed table
rows = []
for c4 in range(4):
    mask = labels4 == c4
    cnt = Counter(y_true[mask])
    n = mask.sum()
    rows.append({
        'Sub-Cluster': f'SC-{c4}',
        'Parent (k=2)': f'C{sub_map[c4]}',
        'N': n,
        'Sehat (0)': cnt.get(0, 0),
        'Sakit (1)': cnt.get(1, 0),
        'Purity (%)': round(max(cnt.values()) / n * 100, 1),
        'Dominan': 'Sehat' if cnt.get(0,0) >= cnt.get(1,0) else 'Sakit'
    })
df_sub = pd.DataFrame(rows).sort_values('Parent (k=2)')

# Feature means per sub-cluster (un-scaled)
sub_profiles = X_raw.copy()
sub_profiles['sub'] = labels4
profile_means = sub_profiles.groupby('sub').mean().round(2)
profile_means.index = [f'SC-{i}' for i in profile_means.index]

# Silhouette samples for k=2
sil_samples2 = silhouette_samples(X, labels2)

print("=== DONE ===")
print("Best k (silhouette):", best_k_sil)
print("\nMixing table (k=2):")
print(mix)
print("\nPurity k=2:", purity)
print("\nSub-cluster table:")
print(df_sub.to_string())

# ── 5. SAVE ALL FIGURES ──────────────────────────────────────────────────────
# ─── FIGURE 1: Overview Dashboard ───────────────────────────────────────────
fig = plt.figure(figsize=(20, 22), facecolor='#0f172a')
gs = gridspec.GridSpec(4, 3, figure=fig, hspace=0.45, wspace=0.35,
                       left=0.07, right=0.95, top=0.93, bottom=0.05)

DARK  = '#0f172a'
CARD  = '#1e293b'
LINE  = '#334155'
ACC1  = '#38bdf8'   # blue
ACC2  = '#fb7185'   # red/pink
ACC3  = '#a78bfa'   # purple
ACC4  = '#34d399'   # green
ACC5  = '#fbbf24'   # amber
WHITE = '#f1f5f9'
GRAY  = '#94a3b8'
CLUSTER_COLORS = [ACC1, ACC2, ACC3, ACC4]

def card_bg(ax, color=CARD):
    ax.set_facecolor(color)
    for spine in ax.spines.values():
        spine.set_edgecolor(LINE)
        spine.set_linewidth(0.8)

def title_ax(ax, txt, size=11):
    ax.set_title(txt, color=WHITE, fontsize=size, fontweight='bold', pad=10)

# ── 5a. Elbow ────────────────────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
card_bg(ax1)
ks = list(K_range)
ax1.plot(ks, inertias, 'o-', color=ACC1, lw=2.5, ms=7, mfc=WHITE)
ax1.fill_between(ks, inertias, alpha=0.12, color=ACC1)
ax1.axvline(best_k_sil, color=ACC5, lw=1.5, ls='--', alpha=0.8)
ax1.set_xlabel('Jumlah Cluster (k)', color=GRAY, fontsize=9)
ax1.set_ylabel('Inertia (WCSS)', color=GRAY, fontsize=9)
ax1.tick_params(colors=GRAY)
title_ax(ax1, '① Elbow Method')
ax1.text(best_k_sil+0.1, max(inertias)*0.95, f'k={best_k_sil}', color=ACC5, fontsize=9)

# ── 5b. Silhouette scores ────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
card_bg(ax2)
bar_colors = [ACC5 if k==best_k_sil else ACC3 for k in ks]
bars = ax2.bar(ks, sil_scores, color=bar_colors, edgecolor='none', width=0.6)
ax2.set_xlabel('Jumlah Cluster (k)', color=GRAY, fontsize=9)
ax2.set_ylabel('Silhouette Score', color=GRAY, fontsize=9)
ax2.tick_params(colors=GRAY)
title_ax(ax2, '② Silhouette Score per k')
ax2.axhline(max(sil_scores), color=ACC5, lw=1, ls='--', alpha=0.6)
for bar, s in zip(bars, sil_scores):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003,
             f'{s:.3f}', ha='center', va='bottom', color=WHITE, fontsize=7.5)

# ── 5c. PCA scatter k=2 ─────────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[0, 2])
card_bg(ax3)
colors_k2 = [ACC1 if l==0 else ACC2 for l in labels2]
ax3.scatter(X_pca[:,0], X_pca[:,1], c=colors_k2, s=25, alpha=0.75, edgecolors='none')
# centroids
c_pca = pca.transform(km2.cluster_centers_)
ax3.scatter(c_pca[:,0], c_pca[:,1], c=[ACC1,ACC2], s=180, marker='*',
            edgecolors=WHITE, linewidths=0.8, zorder=5)
ax3.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)', color=GRAY, fontsize=9)
ax3.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)', color=GRAY, fontsize=9)
ax3.tick_params(colors=GRAY)
patches = [mpatches.Patch(color=ACC1, label=f'Cluster 0'),
           mpatches.Patch(color=ACC2, label=f'Cluster 1')]
ax3.legend(handles=patches, fontsize=8, facecolor=CARD, edgecolor=LINE,
           labelcolor=WHITE, loc='upper right')
title_ax(ax3, '③ PCA Projection (k=2)')

# ── 5d. Mixing heatmap ───────────────────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 0:2])
card_bg(ax4)
mix_vals = mix.values
im = ax4.imshow(mix_vals, cmap='YlOrRd', aspect='auto', vmin=0)
for i in range(mix_vals.shape[0]):
    for j in range(mix_vals.shape[1]):
        pct = mix_vals[i,j] / mix_vals[i].sum() * 100
        ax4.text(j, i, f'{mix_vals[i,j]}\n({pct:.0f}%)',
                 ha='center', va='center', color='#0f172a', fontsize=13, fontweight='bold')
ax4.set_xticks([0,1])
ax4.set_yticks([0,1])
ax4.set_xticklabels(['True: Sehat (0)', 'True: Sakit (1)'], color=WHITE, fontsize=10)
ax4.set_yticklabels(['Cluster 0', 'Cluster 1'], color=WHITE, fontsize=10)
ax4.set_xlabel('Ground Truth', color=GRAY)
ax4.set_ylabel('K-Means Label', color=GRAY)
title_ax(ax4, '④ Mixing Matrix (k=2) — Pencampuran Instance', 13)
cbar = fig.colorbar(im, ax=ax4, pad=0.02)
cbar.ax.yaxis.set_tick_params(color=GRAY)
plt.setp(cbar.ax.yaxis.get_ticklabels(), color=GRAY)

# Purity annotation
for i, p in enumerate(purity):
    ax4.annotate(f'Purity: {p:.1f}%', xy=(1.18, i), xycoords='data',
                 fontsize=11, color=ACC5, fontweight='bold')

# ── 5e. Silhouette plot k=2 ──────────────────────────────────────────────────
ax5 = fig.add_subplot(gs[1, 2])
card_bg(ax5)
y_lower = 10
for i in range(2):
    ith_sil = np.sort(sil_samples2[labels2 == i])
    size_i = ith_sil.shape[0]
    y_upper = y_lower + size_i
    color = [ACC1, ACC2][i]
    ax5.fill_betweenx(np.arange(y_lower, y_upper), 0, ith_sil,
                      facecolor=color, edgecolor='none', alpha=0.85)
    ax5.text(-0.05, y_lower + 0.5*size_i, f'C{i}', color=color, fontsize=9, va='center')
    y_lower = y_upper + 10
ax5.axvline(silhouette_score(X, labels2), color=ACC5, lw=1.5, ls='--')
ax5.set_xlabel('Silhouette Coefficient', color=GRAY, fontsize=9)
ax5.set_yticks([])
ax5.tick_params(colors=GRAY)
title_ax(ax5, '⑤ Silhouette Plot (k=2)')
ax5.text(silhouette_score(X, labels2)+0.01, y_lower*0.5,
         f'avg={silhouette_score(X, labels2):.3f}', color=ACC5, fontsize=8)

# ── 5f. PCA k=4 ──────────────────────────────────────────────────────────────
ax6 = fig.add_subplot(gs[2, 0])
card_bg(ax6)
for ci, col in enumerate(CLUSTER_COLORS):
    mask = labels4 == ci
    ax6.scatter(X_pca[mask,0], X_pca[mask,1], c=col, s=20, alpha=0.7,
                edgecolors='none', label=f'SC-{ci}')
c4_pca = pca.transform(km4.cluster_centers_)
ax6.scatter(c4_pca[:,0], c4_pca[:,1], c=CLUSTER_COLORS, s=160, marker='*',
            edgecolors=WHITE, linewidths=0.8, zorder=5)
ax6.set_xlabel(f'PC1', color=GRAY, fontsize=9)
ax6.set_ylabel(f'PC2', color=GRAY, fontsize=9)
ax6.tick_params(colors=GRAY)
ax6.legend(fontsize=8, facecolor=CARD, edgecolor=LINE, labelcolor=WHITE)
title_ax(ax6, '⑥ PCA Projection (k=4 Sub-cluster)')

# ── 5g. Sub-cluster bar chart ────────────────────────────────────────────────
ax7 = fig.add_subplot(gs[2, 1])
card_bg(ax7)
x = np.arange(4)
w = 0.35
sehat = df_sub['Sehat (0)'].values
sakit = df_sub['Sakit (1)'].values
b1 = ax7.bar(x-w/2, sehat, w, label='Sehat (0)', color=ACC4, edgecolor='none')
b2 = ax7.bar(x+w/2, sakit, w, label='Sakit (1)', color=ACC2, edgecolor='none')
ax7.set_xticks(x)
ax7.set_xticklabels(df_sub['Sub-Cluster'].values, color=WHITE, fontsize=10)
ax7.tick_params(colors=GRAY)
ax7.set_ylabel('Jumlah Instance', color=GRAY, fontsize=9)
ax7.legend(fontsize=9, facecolor=CARD, edgecolor=LINE, labelcolor=WHITE)
title_ax(ax7, '⑦ Komposisi Sub-Cluster (k=4)')
for bar in list(b1)+list(b2):
    h = bar.get_height()
    if h > 0:
        ax7.text(bar.get_x()+bar.get_width()/2, h+1, str(int(h)),
                 ha='center', color=WHITE, fontsize=8)

# ── 5h. Purity per sub-cluster ───────────────────────────────────────────────
ax8 = fig.add_subplot(gs[2, 2])
card_bg(ax8)
purities = df_sub['Purity (%)'].values
sc_names = df_sub['Sub-Cluster'].values
bar_c = [CLUSTER_COLORS[int(s.split('-')[1])] for s in sc_names]
bars8 = ax8.barh(sc_names, purities, color=bar_c, edgecolor='none', height=0.55)
ax8.axvline(50, color=LINE, lw=1, ls='--')
ax8.set_xlim(0, 105)
ax8.tick_params(colors=GRAY)
ax8.set_xlabel('Purity (%)', color=GRAY, fontsize=9)
for bar, p in zip(bars8, purities):
    ax8.text(p+1, bar.get_y()+bar.get_height()/2,
             f'{p}%', va='center', color=WHITE, fontsize=10, fontweight='bold')
title_ax(ax8, '⑧ Purity per Sub-Cluster (k=4)')

# ── 5i. Feature radar / profile heatmap ─────────────────────────────────────
ax9 = fig.add_subplot(gs[3, :])
card_bg(ax9)
# Normalize profiles 0-1 per feature
pm_norm = (profile_means - profile_means.min()) / (profile_means.max() - profile_means.min() + 1e-9)
im9 = ax9.imshow(pm_norm.values, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
ax9.set_xticks(range(len(feature_names)))
ax9.set_xticklabels(feature_names, color=WHITE, fontsize=8, rotation=35, ha='right')
ax9.set_yticks(range(4))
ax9.set_yticklabels(profile_means.index, color=WHITE, fontsize=10)
# Annotate with actual values
for i in range(4):
    for j in range(len(feature_names)):
        val = profile_means.iloc[i, j]
        ax9.text(j, i, f'{val:.1f}', ha='center', va='center',
                 color='#0f172a', fontsize=7.5, fontweight='bold')
cbar9 = fig.colorbar(im9, ax=ax9, pad=0.01, orientation='vertical', fraction=0.015)
cbar9.ax.yaxis.set_tick_params(color=GRAY)
plt.setp(cbar9.ax.yaxis.get_ticklabels(), color=GRAY)
title_ax(ax9, '⑨ Feature Profile Heatmap per Sub-Cluster (Normalized)', 12)

# Main title
fig.text(0.5, 0.965, 'K-Means Clustering — Heart Disease Dataset',
         ha='center', color=WHITE, fontsize=16, fontweight='bold')
fig.text(0.5, 0.955, f'n={len(df)} pasien · {len(feature_names)} fitur · Optimal k={best_k_sil} (Silhouette)',
         ha='center', color=GRAY, fontsize=10)

# ── PERUBAHAN DI SINI ────────────────────────────────────────────────────────
# Tentukan path output gambar
output_dir = '/mnt/user-data/outputs/'
output_file = os.path.join(output_dir, 'kmeans_heart_analysis.png')

# Cek apakah folder ada, jika tidak, buat foldernya terlebih dahulu
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"Direktori {output_dir} berhasil dibuat.")

plt.savefig(output_file, dpi=150, bbox_inches='tight', facecolor=DARK)
plt.close()
print(f"Figure berhasil disimpan di: {output_file}")
# ─────────────────────────────────────────────────────────────────────────────

print("\nSilhouette scores:", dict(zip(ks, [round(s,4) for s in sil_scores])))
print("Inertias:", dict(zip(ks, [round(v,1) for v in inertias])))

=== DONE ===
Best k (silhouette): 2

Mixing table (k=2):
           True:Sehat(0)  True:Sakit(1)
Cluster 0            137             31
Cluster 1             13             89

Purity k=2: [np.float64(81.54761904761905), np.float64(87.25490196078431)]

Sub-cluster table:
  Sub-Cluster Parent (k=2)   N  Sehat (0)  Sakit (1)  Purity (%) Dominan
1        SC-1           C0  36         23         13        63.9   Sehat
2        SC-2           C0  60         53          7        88.3   Sehat
3        SC-3           C0  95         66         29        69.5   Sehat
0        SC-0           C1  79          8         71        89.9   Sakit
Direktori /mnt/user-data/outputs/ berhasil dibuat.
Figure berhasil disimpan di: /mnt/user-data/outputs/kmeans_heart_analysis.png

Silhouette scores: {2: np.float64(0.1719), 3: np.float64(0.1276), 4: np.float64(0.1313), 5: np.float64(0.1044), 6: np.float64(0.1127), 7: np.float64(0.108), 8: np.float64(0.0952), 9: np.float64(0.1046), 10: np.float64(0.0997)}
Inert